In [1]:
import os
from dotenv import load_dotenv
import json
import pathlib
import textwrap
import google.generativeai as genai
from IPython.display import display
from IPython.display import Markdown
from google.api_core import retry
import typing_extensions as typing

def to_markdown(text):
    text = text.replace('•', '  *')
    return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

load_dotenv()
google_api_key = os.getenv('GEMINI_API_KEY')

genai.configure(api_key=google_api_key)

system_prompt ={"text":"you are an medical AI agent.Respond the reaction between the drugs to human when taken together in 100 words"}

# tool to add medicine names to a list
medicine_list = []
def medicine_tool(medicine):
    med = medicine.lower().strip()
    if med not in medicine_list:
        medicine_list.append(med)
    return medicine_list
    
medicine_class={
    "name": "medicine_tool",
    "description":"gets the list of medicines",
    "parameters":{
        "type":"object",
        "properties":{
            "medicine":{
                "type":"string",
                "description":"The medicine name",
            },
        },
        "required":["medicine"],
        "additionalProperties":False
    }
}
tool_config = {
    "function_calling_config": {
        "mode": "ANY",
        "allowed_function_names": ["medicine_class"]
    }
}
user_prompt=f"what is the reaction between the drugs in the list:\n"
user_prompt+=str([medicine_class])

def message_gemini(*user_prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    gemini = genai.GenerativeModel(
    model_name='gemini-1.5-flash',
    system_instruction=system_prompt,
    tool_config=tool_config
)
    response = gemini.generate_content(user_prompt)
    return response.text

In [ ]:
to_markdown(message_gemini("paracetamol","flexon","crosin"))

> Combining paracetamol (acetaminophen) and flurbiprofen (a nonsteroidal anti-inflammatory drug, NSAID) can increase the risk of gastrointestinal bleeding and ulcers, especially with long-term use or high doses.  Paracetamol can cause liver damage at high doses, while flurbiprofen can increase the risk of kidney problems and cardiovascular events.  Taking them together may enhance these risks.  Always consult a doctor before combining medications, as interactions can vary based on individual health conditions and other medications being taken.  This information is not a substitute for professional medical advice.


: 